# Custom Faster R-CNN Debug Notebook

Notebook này dựng custom Faster R-CNN theo từng component để debug độc lập.

Quy ước:
- Không dùng detector hoàn chỉnh từ `torchvision.models.detection`.
- Chỉ dùng ResNet pretrained ImageNet làm backbone.
- Dùng `torchvision.ops.roi_align` và `torchvision.ops.nms` như primitive ops.
- Mỗi nhóm component có smoke test shape/loss để phát hiện lỗi sớm.

In [1]:
from __future__ import annotations

import gc
import math
import os
import sys
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch
from torch import nn
from torch.nn import functional as F
from torchvision.models import ResNet50_Weights, ResNet101_Weights, resnet50, resnet101
from torchvision.ops import nms as tv_nms
from torchvision.ops import roi_align

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from utils.dataset import OdDataset, collate_fn
except Exception as exc:
    OdDataset = None
    collate_fn = None
    print(f"Dataset import skipped: {exc}")

torch.set_float32_matmul_precision("high")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

device: cpu


In [2]:
@dataclass
class DebugConfig:
    num_classes: int = 6  # background + 5 classes
    backbone_name: str = "resnet50"
    pretrained_backbone: bool = True
    trainable_backbone_layers: int = 2
    min_size: int = 512
    max_size: int = 768
    anchor_sizes: tuple[int, ...] = (64, 128, 192, 256, 512)
    anchor_ratios: tuple[float, ...] = (0.33, 0.5, 1.0, 2.0)
    rpn_batch_size: int = 256
    rpn_positive_fraction: float = 0.5
    roi_batch_size: int = 128
    roi_positive_fraction: float = 0.25
    train_pre_nms_top_n: int = 1000
    train_post_nms_top_n: int = 300
    test_pre_nms_top_n: int = 600
    test_post_nms_top_n: int = 100
    box_score_thresh: float = 0.05
    box_nms_thresh: float = 0.5
    roi_channels: int = 256

cfg = DebugConfig()
print(cfg)

DebugConfig(num_classes=6, backbone_name='resnet50', pretrained_backbone=True, trainable_backbone_layers=2, min_size=512, max_size=768, anchor_sizes=(64, 128, 192, 256, 512), anchor_ratios=(0.33, 0.5, 1.0, 2.0), rpn_batch_size=256, rpn_positive_fraction=0.5, roi_batch_size=128, roi_positive_fraction=0.25, train_pre_nms_top_n=1000, train_post_nms_top_n=300, test_pre_nms_top_n=600, test_post_nms_top_n=100, box_score_thresh=0.05, box_nms_thresh=0.5, roi_channels=256)


## 1. Box Utilities

Các hàm box là nền của matching, regression, NMS và metric. Nếu phần này sai thì loss/prediction sẽ sai dây chuyền.

In [3]:
def box_area(boxes: torch.Tensor) -> torch.Tensor:
    return (boxes[:, 2] - boxes[:, 0]).clamp(min=0) * (boxes[:, 3] - boxes[:, 1]).clamp(min=0)


def box_iou(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return boxes1.new_zeros((boxes1.shape[0], boxes2.shape[0]))
    lt = torch.maximum(boxes1[:, None, :2], boxes2[:, :2])
    rb = torch.minimum(boxes1[:, None, 2:], boxes2[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = box_area(boxes1)[:, None] + box_area(boxes2) - inter
    return inter / union.clamp(min=1e-6)


def clip_boxes_to_image(boxes: torch.Tensor, size: tuple[int, int]) -> torch.Tensor:
    height, width = size
    boxes = boxes.clone()
    boxes[:, 0::2] = boxes[:, 0::2].clamp(min=0, max=width)
    boxes[:, 1::2] = boxes[:, 1::2].clamp(min=0, max=height)
    return boxes


def remove_small_boxes(boxes: torch.Tensor, min_size: float) -> torch.Tensor:
    widths = boxes[:, 2] - boxes[:, 0]
    heights = boxes[:, 3] - boxes[:, 1]
    return torch.where((widths >= min_size) & (heights >= min_size))[0]


def encode_boxes(reference_boxes: torch.Tensor, proposals: torch.Tensor) -> torch.Tensor:
    widths = (proposals[:, 2] - proposals[:, 0]).clamp(min=1e-6)
    heights = (proposals[:, 3] - proposals[:, 1]).clamp(min=1e-6)
    ctr_x = proposals[:, 0] + 0.5 * widths
    ctr_y = proposals[:, 1] + 0.5 * heights

    gt_widths = (reference_boxes[:, 2] - reference_boxes[:, 0]).clamp(min=1e-6)
    gt_heights = (reference_boxes[:, 3] - reference_boxes[:, 1]).clamp(min=1e-6)
    gt_ctr_x = reference_boxes[:, 0] + 0.5 * gt_widths
    gt_ctr_y = reference_boxes[:, 1] + 0.5 * gt_heights

    return torch.stack(
        [
            (gt_ctr_x - ctr_x) / widths,
            (gt_ctr_y - ctr_y) / heights,
            torch.log(gt_widths / widths),
            torch.log(gt_heights / heights),
        ],
        dim=1,
    )


def decode_boxes(deltas: torch.Tensor, boxes: torch.Tensor) -> torch.Tensor:
    widths = (boxes[:, 2] - boxes[:, 0]).clamp(min=1e-6)
    heights = (boxes[:, 3] - boxes[:, 1]).clamp(min=1e-6)
    ctr_x = boxes[:, 0] + 0.5 * widths
    ctr_y = boxes[:, 1] + 0.5 * heights

    dx = deltas[:, 0].clamp(min=-10, max=10)
    dy = deltas[:, 1].clamp(min=-10, max=10)
    dw = deltas[:, 2].clamp(min=-5, max=5)
    dh = deltas[:, 3].clamp(min=-5, max=5)

    pred_ctr_x = dx * widths + ctr_x
    pred_ctr_y = dy * heights + ctr_y
    pred_w = torch.exp(dw) * widths
    pred_h = torch.exp(dh) * heights
    return torch.stack(
        [
            pred_ctr_x - 0.5 * pred_w,
            pred_ctr_y - 0.5 * pred_h,
            pred_ctr_x + 0.5 * pred_w,
            pred_ctr_y + 0.5 * pred_h,
        ],
        dim=1,
    )

In [4]:
boxes = torch.tensor([[0., 0., 10., 10.], [5., 5., 15., 15.]])
gt = torch.tensor([[0., 0., 10., 10.], [0., 0., 20., 20.]])
ious = box_iou(boxes, gt)
encoded = encode_boxes(gt, boxes)
decoded = decode_boxes(encoded, boxes)
print("IoU:\n", ious)
print("encoded:\n", encoded)
print("decode close:", torch.allclose(decoded, gt, atol=1e-4))
assert ious.shape == (2, 2)
assert torch.allclose(decoded, gt, atol=1e-4)

IoU:
 tensor([[1.0000, 0.2500],
        [0.1429, 0.2500]])
encoded:
 tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6931, 0.6931]])
decode close: True


## 2. Transform, Resize, Padding

Faster R-CNN resize ảnh theo aspect ratio, scale box cùng hệ tọa độ, rồi pad batch về cùng kích thước.

In [5]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def resize_image_and_boxes(
    image: torch.Tensor,
    boxes: torch.Tensor | None,
    min_size: int,
    max_size: int,
) -> tuple[torch.Tensor, torch.Tensor | None, float]:
    _, height, width = image.shape
    short_side = min(height, width)
    long_side = max(height, width)
    scale = min_size / short_side
    if long_side * scale > max_size:
        scale = max_size / long_side
    new_height = int(round(height * scale))
    new_width = int(round(width * scale))
    image = F.interpolate(
        image.unsqueeze(0),
        size=(new_height, new_width),
        mode="bilinear",
        align_corners=False,
    ).squeeze(0)
    if boxes is not None:
        boxes = boxes * scale
    return image, boxes, scale


def pad_images(images: list[torch.Tensor], size_divisible: int = 16) -> tuple[torch.Tensor, list[tuple[int, int]]]:
    image_sizes = [(image.shape[-2], image.shape[-1]) for image in images]
    max_height = max(size[0] for size in image_sizes)
    max_width = max(size[1] for size in image_sizes)
    max_height = int(math.ceil(max_height / size_divisible) * size_divisible)
    max_width = int(math.ceil(max_width / size_divisible) * size_divisible)
    batch = images[0].new_zeros((len(images), 3, max_height, max_width))
    for index, image in enumerate(images):
        _, height, width = image.shape
        batch[index, :, :height, :width] = image
    return batch, image_sizes


def transform_images(
    images: list[torch.Tensor],
    targets: list[dict[str, torch.Tensor]] | None,
    min_size: int,
    max_size: int,
) -> tuple[torch.Tensor, list[tuple[int, int]], list[tuple[int, int]], list[dict[str, torch.Tensor]] | None]:
    normalized = []
    original_sizes = []
    resized_sizes = []
    new_targets = [] if targets is not None else None
    for idx, image in enumerate(images):
        original_sizes.append((image.shape[-2], image.shape[-1]))
        boxes = targets[idx]["boxes"] if targets is not None else None
        resized, resized_boxes, _ = resize_image_and_boxes(image, boxes, min_size, max_size)
        normalized.append((resized - IMAGENET_MEAN.to(image.device)) / IMAGENET_STD.to(image.device))
        resized_sizes.append((resized.shape[-2], resized.shape[-1]))
        if new_targets is not None:
            target = {k: v for k, v in targets[idx].items()}
            target["boxes"] = resized_boxes
            new_targets.append(target)
    batch, _ = pad_images(normalized)
    return batch, original_sizes, resized_sizes, new_targets

In [6]:
image = torch.rand(3, 320, 480)
target = {"boxes": torch.tensor([[10., 20., 100., 200.]]), "labels": torch.tensor([1])}
batch, original_sizes, resized_sizes, new_targets = transform_images([image], [target], 512, 768)
print("batch:", tuple(batch.shape))
print("original:", original_sizes, "resized:", resized_sizes)
print("scaled boxes:", new_targets[0]["boxes"])
assert batch.ndim == 4
assert new_targets[0]["boxes"].shape == (1, 4)

batch: (1, 3, 512, 768)
original: [(320, 480)] resized: [(512, 768)]
scaled boxes: tensor([[ 16.,  32., 160., 320.]])


## 3. Backbone ResNet ImageNet

Backbone trả về một feature map duy nhất từ `layer3` stride khoảng 16. Đây là bản đơn giản để debug trước khi cân nhắc FPN.

In [7]:
BACKBONE_WEIGHTS = {
    "resnet50": ResNet50_Weights.DEFAULT,
    "resnet101": ResNet101_Weights.DEFAULT,
}
BACKBONE_FACTORIES = {
    "resnet50": resnet50,
    "resnet101": resnet101,
}
BACKBONE_OUT_CHANNELS = {
    "resnet50": 1024,
    "resnet101": 1024,
}


class ResNetBackbone(nn.Module):
    def __init__(
        self,
        backbone_name: str = "resnet50",
        pretrained_backbone: bool = True,
        trainable_backbone_layers: int = 2,
    ) -> None:
        super().__init__()
        weights = BACKBONE_WEIGHTS[backbone_name] if pretrained_backbone else None
        backbone = BACKBONE_FACTORIES[backbone_name](weights=weights)
        self.body = nn.Sequential(
            OrderedDict(
                [
                    ("conv1", backbone.conv1),
                    ("bn1", backbone.bn1),
                    ("relu", backbone.relu),
                    ("maxpool", backbone.maxpool),
                    ("layer1", backbone.layer1),
                    ("layer2", backbone.layer2),
                    ("layer3", backbone.layer3),
                ]
            )
        )
        self.out_channels = BACKBONE_OUT_CHANNELS[backbone_name]
        if pretrained_backbone:
            for p in backbone.conv1.parameters():
                p.requires_grad_(False)
            for p in backbone.bn1.parameters():
                p.requires_grad_(False)
            layers = [backbone.layer1, backbone.layer2, backbone.layer3]
            frozen_layers = layers[: max(0, len(layers) - trainable_backbone_layers)]
            for layer in frozen_layers:
                for p in layer.parameters():
                    p.requires_grad_(False)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.body(images)


def count_parameters(module: nn.Module) -> dict[str, int]:
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable}

In [8]:
backbone = ResNetBackbone(cfg.backbone_name, pretrained_backbone=False).to(device)
with torch.no_grad():
    features = backbone(torch.rand(2, 3, 512, 768, device=device))
print("features:", tuple(features.shape))
print(count_parameters(backbone))
assert features.shape[1] == 1024
del backbone, features
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

features: (2, 1024, 32, 48)
{'total': 8543296, 'trainable': 8543296}


## 4. Anchors và RPN

Anchor generator tạo `num_locations x num_sizes x num_ratios` anchors. RPN head dự đoán objectness và bbox delta cho từng anchor.

In [9]:
class CustomAnchorGenerator(nn.Module):
    def __init__(self, sizes: tuple[int, ...], ratios: tuple[float, ...]) -> None:
        super().__init__()
        anchors = []
        for size in sizes:
            area = float(size * size)
            for ratio in ratios:
                width = math.sqrt(area * ratio)
                height = area / width
                anchors.append([-0.5 * width, -0.5 * height, 0.5 * width, 0.5 * height])
        self.sizes = sizes
        self.ratios = ratios
        self.register_buffer("base_anchors", torch.tensor(anchors, dtype=torch.float32))

    @property
    def num_anchors(self) -> int:
        return int(self.base_anchors.shape[0])

    def forward(self, feature: torch.Tensor, image_size: tuple[int, int]) -> torch.Tensor:
        _, _, feature_height, feature_width = feature.shape
        image_height, image_width = image_size
        stride_y = image_height / feature_height
        stride_x = image_width / feature_width
        shifts_x = (torch.arange(feature_width, device=feature.device, dtype=torch.float32) + 0.5) * stride_x
        shifts_y = (torch.arange(feature_height, device=feature.device, dtype=torch.float32) + 0.5) * stride_y
        shift_y, shift_x = torch.meshgrid(shifts_y, shifts_x, indexing="ij")
        shifts = torch.stack(
            [shift_x.reshape(-1), shift_y.reshape(-1), shift_x.reshape(-1), shift_y.reshape(-1)],
            dim=1,
        )
        return (shifts[:, None, :] + self.base_anchors[None, :, :]).reshape(-1, 4)


class RPNHead(nn.Module):
    def __init__(self, in_channels: int, num_anchors: int) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.objectness = nn.Conv2d(in_channels, num_anchors, kernel_size=1)
        self.bbox_reg = nn.Conv2d(in_channels, num_anchors * 4, kernel_size=1)
        for layer in [self.conv, self.objectness, self.bbox_reg]:
            nn.init.normal_(layer.weight, std=0.01)
            nn.init.constant_(layer.bias, 0)

    def forward(self, feature: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        hidden = F.relu(self.conv(feature))
        objectness = self.objectness(hidden)
        bbox_reg = self.bbox_reg(hidden)
        batch_size, anchors, height, width = objectness.shape
        objectness = objectness.permute(0, 2, 3, 1).reshape(batch_size, -1)
        bbox_reg = bbox_reg.view(batch_size, anchors, 4, height, width)
        bbox_reg = bbox_reg.permute(0, 3, 4, 1, 2).reshape(batch_size, -1, 4)
        return objectness, bbox_reg

In [10]:
feature = torch.rand(2, 1024, 32, 48, device=device)
anchor_gen = CustomAnchorGenerator(cfg.anchor_sizes, cfg.anchor_ratios).to(device)
rpn = RPNHead(1024, anchor_gen.num_anchors).to(device)
anchors = anchor_gen(feature, (512, 768))
objectness, deltas = rpn(feature)
print("anchors:", tuple(anchors.shape), "per location:", anchor_gen.num_anchors)
print("objectness:", tuple(objectness.shape), "deltas:", tuple(deltas.shape))
assert anchors.shape[0] == 32 * 48 * anchor_gen.num_anchors
assert objectness.shape[:2] == (2, anchors.shape[0])
del feature, anchor_gen, rpn, anchors, objectness, deltas
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

anchors: (30720, 4) per location: 20
objectness: (2, 30720) deltas: (2, 30720, 4)


## 5. Matching, Sampling và Proposal Generation

Cell này chứa logic target assignment cho RPN và tạo proposal. Đây là nơi thường gây lỗi memory/time nếu số proposal quá lớn.

In [11]:
def sample_labels(labels: torch.Tensor, batch_size: int, positive_fraction: float) -> torch.Tensor:
    positive = torch.where(labels == 1)[0]
    negative = torch.where(labels == 0)[0]
    num_positive = min(int(batch_size * positive_fraction), positive.numel())
    num_negative = min(batch_size - num_positive, negative.numel())
    perm_pos = positive[torch.randperm(positive.numel(), device=labels.device)[:num_positive]]
    perm_neg = negative[torch.randperm(negative.numel(), device=labels.device)[:num_negative]]
    return torch.cat([perm_pos, perm_neg], dim=0)


def assign_rpn_targets(
    anchors: torch.Tensor,
    targets: list[dict[str, torch.Tensor]],
    image_sizes: list[tuple[int, int]],
) -> tuple[list[torch.Tensor], list[torch.Tensor]]:
    labels = []
    regression_targets = []
    for target, image_size in zip(targets, image_sizes):
        anchors_in_image = clip_boxes_to_image(anchors, image_size)
        gt_boxes = target["boxes"]
        label = torch.full((anchors.shape[0],), -1.0, device=anchors.device)
        matched_gt = torch.zeros_like(anchors)
        if gt_boxes.numel() == 0:
            label[:] = 0
        else:
            ious = box_iou(anchors_in_image, gt_boxes)
            max_iou, matched_idx = ious.max(dim=1)
            label[max_iou < 0.3] = 0
            label[max_iou >= 0.7] = 1
            best_per_gt = ious.argmax(dim=0)
            label[best_per_gt] = 1
            matched_gt = gt_boxes[matched_idx]
        labels.append(label)
        regression_targets.append(encode_boxes(matched_gt, anchors_in_image))
    return labels, regression_targets


def rpn_losses(
    objectness: torch.Tensor,
    pred_bbox_deltas: torch.Tensor,
    anchors: torch.Tensor,
    targets: list[dict[str, torch.Tensor]],
    image_sizes: list[tuple[int, int]],
    cfg: DebugConfig,
) -> tuple[torch.Tensor, torch.Tensor]:
    labels, regression_targets = assign_rpn_targets(anchors, targets, image_sizes)
    objectness_loss = objectness.sum() * 0
    box_loss = pred_bbox_deltas.sum() * 0
    for image_index, labels_per_image in enumerate(labels):
        sampled = sample_labels(labels_per_image, cfg.rpn_batch_size, cfg.rpn_positive_fraction)
        sampled_labels = labels_per_image[sampled]
        objectness_loss = objectness_loss + F.binary_cross_entropy_with_logits(
            objectness[image_index][sampled],
            sampled_labels,
        )
        positives = sampled[sampled_labels == 1]
        if positives.numel():
            box_loss = box_loss + F.smooth_l1_loss(
                pred_bbox_deltas[image_index][positives],
                regression_targets[image_index][positives],
                beta=1 / 9,
                reduction="sum",
            ) / max(positives.numel(), 1)
    normalizer = max(len(targets), 1)
    return objectness_loss / normalizer, box_loss / normalizer


def generate_proposals(
    objectness: torch.Tensor,
    pred_bbox_deltas: torch.Tensor,
    anchors: torch.Tensor,
    image_sizes: list[tuple[int, int]],
    cfg: DebugConfig,
    training: bool,
) -> list[torch.Tensor]:
    proposals = []
    pre_nms_top_n = cfg.train_pre_nms_top_n if training else cfg.test_pre_nms_top_n
    post_nms_top_n = cfg.train_post_nms_top_n if training else cfg.test_post_nms_top_n
    for image_index, image_size in enumerate(image_sizes):
        scores = objectness[image_index].sigmoid()
        num_top = min(pre_nms_top_n, scores.numel())
        top_scores, top_idx = scores.topk(num_top)
        boxes = decode_boxes(pred_bbox_deltas[image_index][top_idx], anchors[top_idx])
        boxes = clip_boxes_to_image(boxes, image_size)
        keep = remove_small_boxes(boxes, min_size=2)
        boxes, top_scores = boxes[keep], top_scores[keep]
        keep = tv_nms(boxes, top_scores, 0.7)[:post_nms_top_n]
        proposals.append(boxes[keep].detach())
    return proposals

In [12]:
feature = torch.rand(1, 1024, 32, 48, device=device)
anchor_gen = CustomAnchorGenerator(cfg.anchor_sizes, cfg.anchor_ratios).to(device)
rpn = RPNHead(1024, anchor_gen.num_anchors).to(device)
anchors = anchor_gen(feature, (512, 768))
objectness, deltas = rpn(feature)
targets = [{"boxes": torch.tensor([[80., 90., 260., 330.]], device=device), "labels": torch.tensor([1], device=device)}]
loss_obj, loss_box = rpn_losses(objectness, deltas, anchors, targets, [(512, 768)], cfg)
props = generate_proposals(objectness, deltas, anchors, [(512, 768)], cfg, training=True)
print("rpn losses:", float(loss_obj.detach().cpu()), float(loss_box.detach().cpu()))
print("proposals:", [tuple(p.shape) for p in props])
assert props[0].shape[1] == 4
del feature, anchor_gen, rpn, anchors, objectness, deltas, targets, props
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

rpn losses: 0.697906494140625 0.4163203239440918
proposals: [(300, 4)]


## 6. RoI Align và ROI Head

ROI head nhận proposal, pooling feature, phân loại class và regress bbox theo từng class.

In [13]:
class ROIHead(nn.Module):
    def __init__(self, in_channels: int, num_classes: int, pool_size: int = 7) -> None:
        super().__init__()
        self.pool_size = pool_size
        hidden = 1024
        self.fc1 = nn.Linear(in_channels * pool_size * pool_size, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.cls_score = nn.Linear(hidden, num_classes)
        self.bbox_pred = nn.Linear(hidden, num_classes * 4)
        for layer in [self.fc1, self.fc2, self.cls_score, self.bbox_pred]:
            nn.init.normal_(layer.weight, std=0.01)
            nn.init.constant_(layer.bias, 0)

    def pool(
        self,
        feature: torch.Tensor,
        proposals: list[torch.Tensor],
        image_sizes: list[tuple[int, int]],
    ) -> torch.Tensor:
        _, channels, feature_height, feature_width = feature.shape
        if not any(boxes.numel() for boxes in proposals):
            return feature.new_zeros((0, channels, self.pool_size, self.pool_size))
        rois = []
        for image_index, boxes in enumerate(proposals):
            if boxes.numel() == 0:
                continue
            image_height, image_width = image_sizes[image_index]
            scaled_boxes = boxes.clone()
            scaled_boxes[:, 0::2] *= feature_width / image_width
            scaled_boxes[:, 1::2] *= feature_height / image_height
            batch_indices = torch.full(
                (scaled_boxes.shape[0], 1),
                image_index,
                dtype=scaled_boxes.dtype,
                device=scaled_boxes.device,
            )
            rois.append(torch.cat([batch_indices, scaled_boxes], dim=1))
        return roi_align(
            feature,
            torch.cat(rois, dim=0),
            output_size=(self.pool_size, self.pool_size),
            spatial_scale=1.0,
            sampling_ratio=2,
            aligned=True,
        )

    def forward(
        self,
        feature: torch.Tensor,
        proposals: list[torch.Tensor],
        image_sizes: list[tuple[int, int]],
    ) -> tuple[torch.Tensor, torch.Tensor]:
        pooled = self.pool(feature, proposals, image_sizes)
        if pooled.numel() == 0:
            return pooled.new_zeros((0, self.cls_score.out_features)), pooled.new_zeros((0, self.bbox_pred.out_features))
        x = pooled.flatten(start_dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.cls_score(x), self.bbox_pred(x)


def assign_roi_targets(
    proposals: list[torch.Tensor],
    targets: list[dict[str, torch.Tensor]],
    cfg: DebugConfig,
) -> tuple[list[torch.Tensor], list[torch.Tensor], list[torch.Tensor]]:
    sampled_proposals = []
    labels = []
    regression_targets = []
    for proposals_per_image, target in zip(proposals, targets):
        gt_boxes = target["boxes"]
        gt_labels = target["labels"]
        proposals_per_image = torch.cat([proposals_per_image, gt_boxes], dim=0)
        if gt_boxes.numel() == 0:
            labels_per_image = torch.zeros((proposals_per_image.shape[0],), dtype=torch.long, device=proposals_per_image.device)
            matched_gt = torch.zeros_like(proposals_per_image)
        else:
            ious = box_iou(proposals_per_image, gt_boxes)
            max_iou, matched_idx = ious.max(dim=1)
            labels_per_image = gt_labels[matched_idx]
            labels_per_image[max_iou < 0.5] = 0
            ignore = (max_iou >= 0.0) & (max_iou < 0.1)
            labels_per_image[ignore] = -1
            matched_gt = gt_boxes[matched_idx]
        sampling_labels = (labels_per_image > 0).float().where(
            labels_per_image >= 0,
            torch.tensor(-1.0, device=labels_per_image.device),
        )
        sampled = sample_labels(sampling_labels, cfg.roi_batch_size, cfg.roi_positive_fraction)
        sampled_proposals.append(proposals_per_image[sampled])
        labels.append(labels_per_image[sampled].clamp(min=0))
        regression_targets.append(encode_boxes(matched_gt[sampled], proposals_per_image[sampled]))
    return sampled_proposals, labels, regression_targets


def roi_losses(
    class_logits: torch.Tensor,
    box_regression: torch.Tensor,
    labels: list[torch.Tensor],
    regression_targets: list[torch.Tensor],
    num_classes: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    labels_cat = torch.cat(labels, dim=0)
    regression_targets_cat = torch.cat(regression_targets, dim=0)
    classification_loss = F.cross_entropy(class_logits, labels_cat)
    positive = torch.where(labels_cat > 0)[0]
    if positive.numel() == 0:
        return classification_loss, box_regression.sum() * 0
    box_regression = box_regression.reshape(box_regression.shape[0], num_classes, 4)
    box_loss = F.smooth_l1_loss(
        box_regression[positive, labels_cat[positive]],
        regression_targets_cat[positive],
        beta=1 / 9,
        reduction="sum",
    ) / labels_cat.numel()
    return classification_loss, box_loss

In [14]:
roi_head = ROIHead(cfg.roi_channels, cfg.num_classes).to(device)
roi_feature = torch.rand(1, cfg.roi_channels, 32, 48, device=device)
proposals = [torch.tensor([[80., 90., 260., 330.], [10., 10., 40., 40.]], device=device)]
targets = [{"boxes": torch.tensor([[80., 90., 260., 330.]], device=device), "labels": torch.tensor([1], device=device)}]
sampled_props, labels, reg_targets = assign_roi_targets(proposals, targets, cfg)
class_logits, box_reg = roi_head(roi_feature, sampled_props, [(512, 768)])
loss_cls, loss_box = roi_losses(class_logits, box_reg, labels, reg_targets, cfg.num_classes)
print("sampled:", [tuple(p.shape) for p in sampled_props])
print("roi logits:", tuple(class_logits.shape), "box_reg:", tuple(box_reg.shape))
print("roi losses:", float(loss_cls.detach().cpu()), float(loss_box.detach().cpu()))
assert class_logits.shape[0] == sampled_props[0].shape[0]
del roi_head, roi_feature, proposals, targets, sampled_props, labels, reg_targets, class_logits, box_reg
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

sampled: [(2, 4)]
roi logits: (2, 6) box_reg: (2, 24)
roi losses: 1.751977562904358 0.035241879522800446


## 7. Full Custom Faster R-CNN

Cell này ghép các component lại. Khi debug, nên chạy smoke test full model với ảnh nhỏ trước, rồi mới đưa batch thật từ dataset.

In [15]:
class CustomFasterRCNNDebug(nn.Module):
    def __init__(self, cfg: DebugConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.num_classes = cfg.num_classes
        self.backbone = ResNetBackbone(
            cfg.backbone_name,
            pretrained_backbone=cfg.pretrained_backbone,
            trainable_backbone_layers=cfg.trainable_backbone_layers,
        )
        self.anchor_generator = CustomAnchorGenerator(cfg.anchor_sizes, cfg.anchor_ratios)
        self.rpn_head = RPNHead(self.backbone.out_channels, self.anchor_generator.num_anchors)
        self.roi_projection = nn.Conv2d(self.backbone.out_channels, cfg.roi_channels, kernel_size=1)
        self.roi_head = ROIHead(cfg.roi_channels, cfg.num_classes)
        nn.init.normal_(self.roi_projection.weight, std=0.01)
        nn.init.constant_(self.roi_projection.bias, 0)

    def transform(
        self,
        images: list[torch.Tensor],
        targets: list[dict[str, torch.Tensor]] | None = None,
    ) -> tuple[torch.Tensor, list[tuple[int, int]], list[tuple[int, int]], list[dict[str, torch.Tensor]] | None]:
        return transform_images(images, targets, self.cfg.min_size, self.cfg.max_size)

    def postprocess_detections(
        self,
        class_logits: torch.Tensor,
        box_regression: torch.Tensor,
        proposals: list[torch.Tensor],
        image_sizes: list[tuple[int, int]],
        original_sizes: list[tuple[int, int]],
    ) -> list[dict[str, torch.Tensor]]:
        scores = F.softmax(class_logits, dim=-1)
        box_regression = box_regression.reshape(box_regression.shape[0], self.num_classes, 4)
        results = []
        start = 0
        for image_index, boxes in enumerate(proposals):
            num_boxes = boxes.shape[0]
            scores_per_image = scores[start : start + num_boxes]
            deltas_per_image = box_regression[start : start + num_boxes]
            start += num_boxes
            image_boxes = []
            image_scores = []
            image_labels = []
            for class_index in range(1, self.num_classes):
                class_scores = scores_per_image[:, class_index]
                keep = torch.where(class_scores >= self.cfg.box_score_thresh)[0]
                if keep.numel() == 0:
                    continue
                decoded = decode_boxes(deltas_per_image[keep, class_index], boxes[keep])
                decoded = clip_boxes_to_image(decoded, image_sizes[image_index])
                keep_size = remove_small_boxes(decoded, min_size=2)
                decoded = decoded[keep_size]
                kept_scores = class_scores[keep][keep_size]
                keep_nms = tv_nms(decoded, kept_scores, self.cfg.box_nms_thresh)
                image_boxes.append(decoded[keep_nms])
                image_scores.append(kept_scores[keep_nms])
                image_labels.append(torch.full((keep_nms.numel(),), class_index, dtype=torch.long, device=boxes.device))
            if image_boxes:
                final_boxes = torch.cat(image_boxes, dim=0)
                final_scores = torch.cat(image_scores, dim=0)
                final_labels = torch.cat(image_labels, dim=0)
                order = final_scores.argsort(descending=True)[:100]
                final_boxes = final_boxes[order]
                final_scores = final_scores[order]
                final_labels = final_labels[order]
            else:
                final_boxes = boxes.new_zeros((0, 4))
                final_scores = boxes.new_zeros((0,))
                final_labels = boxes.new_zeros((0,), dtype=torch.long)
            resized_h, resized_w = image_sizes[image_index]
            original_h, original_w = original_sizes[image_index]
            final_boxes[:, 0::2] *= original_w / resized_w
            final_boxes[:, 1::2] *= original_h / resized_h
            final_boxes = clip_boxes_to_image(final_boxes, original_sizes[image_index])
            results.append({"boxes": final_boxes, "labels": final_labels, "scores": final_scores})
        return results

    def forward(
        self,
        images: list[torch.Tensor],
        targets: list[dict[str, torch.Tensor]] | None = None,
    ) -> dict[str, torch.Tensor] | list[dict[str, torch.Tensor]]:
        if self.training and targets is None:
            raise ValueError("targets are required in training mode")
        batch, original_sizes, resized_sizes, resized_targets = self.transform(images, targets)
        feature = self.backbone(batch)
        objectness, pred_bbox_deltas = self.rpn_head(feature)
        anchors = self.anchor_generator(feature, batch.shape[-2:])
        proposals = generate_proposals(
            objectness,
            pred_bbox_deltas,
            anchors,
            resized_sizes,
            self.cfg,
            training=self.training,
        )
        roi_feature = F.relu(self.roi_projection(feature))

        if self.training:
            assert resized_targets is not None
            loss_objectness, loss_rpn_box_reg = rpn_losses(
                objectness,
                pred_bbox_deltas,
                anchors,
                resized_targets,
                resized_sizes,
                self.cfg,
            )
            sampled_proposals, labels, regression_targets = assign_roi_targets(proposals, resized_targets, self.cfg)
            class_logits, box_regression = self.roi_head(roi_feature, sampled_proposals, resized_sizes)
            loss_classifier, loss_box_reg = roi_losses(
                class_logits,
                box_regression,
                labels,
                regression_targets,
                self.num_classes,
            )
            return {
                "loss_classifier": loss_classifier,
                "loss_box_reg": loss_box_reg,
                "loss_objectness": loss_objectness,
                "loss_rpn_box_reg": loss_rpn_box_reg,
            }

        class_logits, box_regression = self.roi_head(roi_feature, proposals, resized_sizes)
        return self.postprocess_detections(class_logits, box_regression, proposals, resized_sizes, original_sizes)

In [16]:
small_cfg = DebugConfig(
    num_classes=cfg.num_classes,
    backbone_name="resnet50",
    pretrained_backbone=False,
    min_size=128,
    max_size=192,
    train_pre_nms_top_n=200,
    train_post_nms_top_n=64,
    test_pre_nms_top_n=100,
    test_post_nms_top_n=32,
)
model = CustomFasterRCNNDebug(small_cfg).to(device)
images = [torch.rand(3, 128, 160, device=device)]
targets = [{"boxes": torch.tensor([[20., 25., 90., 110.]], device=device), "labels": torch.tensor([1], device=device)}]
model.train()
losses = model(images, targets)
print({k: float(v.detach().cpu()) for k, v in losses.items()})
loss = sum(losses.values())
loss.backward()
print("backward ok", count_parameters(model))
model.eval()
with torch.no_grad():
    outputs = model(images)
print({k: tuple(v.shape) for k, v in outputs[0].items()})
del model, images, targets, losses, outputs, loss
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

{'loss_classifier': 1.7865626811981201, 'loss_box_reg': 0.044824909418821335, 'loss_objectness': 0.7569770812988281, 'loss_rpn_box_reg': 0.556995153427124}
backward ok {'total': 32272834, 'trainable': 32272834}
{'boxes': (29, 4), 'labels': (29,), 'scores': (29,)}


## 8. Debug với Batch Thật từ Dataset

Cell này chỉ chạy nếu folder `public` tồn tại. Nên dùng `pretrained_backbone=False` khi smoke test để tránh tải weights nếu môi trường không có mạng/cache.

In [17]:
train_ann = PROJECT_ROOT / "public/annotations/train.json"
train_img = PROJECT_ROOT / "public/train/images"
if OdDataset is None or not train_ann.exists():
    print("Dataset smoke test skipped: public dataset not found.")
else:
    dataset = OdDataset(train_ann, train_img)
    image, target = dataset[0]
    print("classes:", dataset.classes)
    print("image:", tuple(image.shape))
    print("target:", {k: tuple(v.shape) if torch.is_tensor(v) else v for k, v in target.items()})

    data_cfg = DebugConfig(num_classes=len(dataset.classes) + 1, pretrained_backbone=False, min_size=256, max_size=384)
    model = CustomFasterRCNNDebug(data_cfg).to(device)
    model.train()
    images = [image.to(device)]
    targets = [{k: v.to(device) for k, v in target.items()}]
    losses = model(images, targets)
    print("losses:", {k: float(v.detach().cpu()) for k, v in losses.items()})
    del model, images, targets, losses, dataset, image, target
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

classes: ['person', 'car', 'dog', 'cat', 'chair']
image: (3, 375, 500)
target: {'boxes': (1, 4), 'labels': (1,), 'image_id': (1,), 'area': (1,), 'iscrowd': (1,)}
losses: {'loss_classifier': 1.7903517484664917, 'loss_box_reg': 0.029519367963075638, 'loss_objectness': 0.8074563145637512, 'loss_rpn_box_reg': 1.2954553365707397}


## 9. Memory Debug Helpers

Dùng các helper này khi nghi ngờ leak. Chạy sau từng cell/batch để xem RSS/CUDA có tăng không.

In [18]:
def memory_summary() -> dict[str, float | str]:
    summary: dict[str, float | str] = {}
    try:
        import resource
        rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        summary["peak_rss_mb"] = rss / 1024 if sys.platform != "darwin" else rss / (1024 ** 2)
    except Exception as exc:
        summary["rss_error"] = str(exc)
    if torch.cuda.is_available():
        summary["cuda_alloc_mb"] = torch.cuda.memory_allocated() / (1024 ** 2)
        summary["cuda_reserved_mb"] = torch.cuda.memory_reserved() / (1024 ** 2)
        summary["cuda_peak_alloc_mb"] = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return summary


def cleanup() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(memory_summary())

{'peak_rss_mb': 1222.265625}
